# A Catholic Introduction to Artificial Intelligence
## Final Project  Module 3: Building and Evaluating the Model
### Classes 5 & 6

---

> *"Important and sensitive decisions  concerning employment, credit, access to public services or even a person's reputation  risk being fully delegated to automated systems that do not know compassion, mercy, forgiveness, and above all, the hope that people are able to change."*
>  Pope Leo XIV, *Magnifica Humanitas*, no. 102

---

## Module Overview

In Module 2 you analyzed the data deeply and made deliberate decisions about which features to include in your model. Now we build it.

This module introduces the full machine learning workflow:
1. **Prepare** the data for modeling
2. **Split** it into training and test sets
3. **Train** two different model types and understand what they are actually doing
4. **Evaluate** the models honestly  and discover that accuracy alone is a dangerously incomplete metric
5. **Interpret** the model's predictions and understand where it fails

By the end of this module, you will have a working predictive model  and a clear-eyed understanding of why a model that is "81% accurate" can still cause significant harm.

---

In [ ]:
%pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Load and prepare data (same as Module 2)
df = pd.read_csv('synthetic_homework_dataset.csv',
                 parse_dates=['date_assigned', 'date_submitted'])

# Engineered features from Module 2
df['time_pressure'] = df['difficulty'] / df['days_until_due']

# Encode the categorical assignment_type column as a number
le = LabelEncoder()
df['assignment_type_enc'] = le.fit_transform(df['assignment_type'])
print('Assignment type encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

print(f'\nData loaded: {len(df)} rows')

---

## Part 1: Preparing the Data for Modeling

Before training any model, we need to make three decisions:
1. **Which features to use**  your decision from Module 2
2. **How to split the data** into training and test sets
3. **Whether to scale the features**  required for logistic regression

### Why Do We Need a Train/Test Split?

If we trained and evaluated the model on the same data, we would be measuring how well it memorized the data  not how well it generalizes to new students and assignments it has never seen. This is like letting a student study the exact exam questions in advance: the score tells you nothing about whether they actually learned the material.

We hold out 20% of the data as a **test set**  the model never sees this data during training. After training, we evaluate performance on the test set only. This gives us an honest estimate of how the model would perform on new data.

In [ ]:
# â”€â”€ FEATURE SELECTION â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# We define two feature sets so we can compare them:
#
# CONSERVATIVE: Uses only assignment properties + one engineered feature
#               No student history  avoids the feedback loop problem
#
# FULL: Also includes prior_completion_rate, prior_avg_grade, and
#       prior_avg_homework_time  stronger predictors, higher ethical stakes
#
# You can also plug in your own feature set from Module 2 decisions.

CONSERVATIVE_FEATURES = [
    'num_questions',
    'difficulty',
    'days_until_due',
    'time_pressure',
    'assignment_type_enc',
]

FULL_FEATURES = CONSERVATIVE_FEATURES + [
    'prior_completion_rate',
    'prior_avg_grade',
    'prior_avg_homework_time',
]

# â”€â”€ CHOOSE WHICH FEATURE SET TO USE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Change this to CONSERVATIVE_FEATURES to compare
SELECTED_FEATURES = CONSERVATIVE_FEATURES

# Target variable
TARGET = 'completed_on_time'

# Build X and y
X = df[SELECTED_FEATURES]
y = df[TARGET]

print(f'Features selected: {SELECTED_FEATURES}')
print(f'X shape: {X.shape}')
print(f'Class balance: {y.value_counts().to_dict()} ({y.mean()*100:.1f}% on time)')

In [ ]:
# â”€â”€ TRAIN / TEST SPLIT â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# random_state=42 ensures everyone gets the same split
# stratify=y ensures the on-time/late ratio is preserved in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set:  {len(X_train)} rows ({y_train.mean()*100:.1f}% on time)')
print(f'Test set:      {len(X_test)} rows ({y_test.mean()*100:.1f}% on time)')
print()
print(f'The model will be trained on {len(X_train)} examples.')
print(f'It will be evaluated on {len(X_test)} examples it has NEVER seen.')

In [ ]:
# â”€â”€ FEATURE SCALING â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Logistic regression is sensitive to feature scale.
# days_until_due ranges from 1â€“6; prior_avg_grade ranges from 60â€“100.
# Without scaling, the larger-scale features would dominate unfairly.
#
# StandardScaler transforms each feature to have mean=0 and std=1.
# IMPORTANT: We fit the scaler on training data only, then apply it to test data.
# Fitting on test data would be "data leakage"  another form of cheating.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)       # transform only, don't re-fit

print('Scaling complete.')
print('Before scaling  days_until_due range:', X_train['days_until_due'].min(),
      'to', X_train['days_until_due'].max())

col_idx = list(X_train.columns).index('days_until_due')
scaled_col = X_train_scaled[:, col_idx]
print(f'After scaling   days_until_due range: {scaled_col.min():.2f} to {scaled_col.max():.2f}')
print(f'After scaling   days_until_due mean: {scaled_col.mean():.4f} (should be ~0)')

---

## Part 2: Training the Models

We will train two different types of model and compare them. Understanding what each model does differently is important for both technical and ethical reasons.

### Model 1: Logistic Regression

Logistic regression is one of the oldest and most widely used classification algorithms. Despite its name, it is a classification model, not a regression model. It works by finding a weighted combination of the input features that best separates the two classes (on time vs. late).

Each feature gets a **coefficient**  a weight  that reflects how much it contributes to the prediction. Positive coefficients push toward "on time"; negative coefficients push toward "late." This makes logistic regression highly **interpretable**: we can see exactly which features matter and in which direction.

### Model 2: Decision Tree

A decision tree learns a sequence of if-then rules. It splits the data repeatedly based on feature thresholds, choosing the split that best separates on-time from late submissions. The result looks like a flowchart.

Decision trees are also highly interpretable  you can literally follow the tree to see why any individual prediction was made.

In [ ]:
# â”€â”€ TRAIN LOGISTIC REGRESSION â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

print('Logistic Regression trained.')
print()

# Show the coefficients  these tell us what the model learned
coef_df = pd.DataFrame({
    'Feature': SELECTED_FEATURES,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print('What the model learned (coefficients):')
print('Positive = pushes toward ON TIME | Negative = pushes toward LATE')
print()
print(coef_df.to_string(index=False))

In [ ]:
# Visualize the coefficients
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4CAF50' if v > 0 else '#E57373' for v in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coefficient Value')
ax.set_title('Logistic Regression: What Did the Model Learn?\n'
             'Green = pushes toward "on time", Red = pushes toward "late"',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.01 if val >= 0 else -0.01),
            bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

### Reading the Coefficients

The coefficient chart reveals what the model actually learned from the data. Notice:

- **`prior_completion_rate`** has by far the largest positive coefficient  a student's past completion rate is the single most influential factor in the model's prediction. This confirms the finding from Module 2: past behavior is the strongest predictor.
- **`days_until_due`** has a negative coefficient — more lead time is associated with *more* lateness, not less. This seems counterintuitive. Can you think of why this might be?
- **`prior_avg_grade`** has a small negative coefficient  somewhat surprising, and worth investigating.

> 🤔 **Think about it:** The model learned that students with more days until the deadline are slightly *more* likely to be late. This is a real pattern in the data — long deadlines are correlated with lower completion rates. Why might that be? What does it tell us about procrastination and human behavior?

In [ ]:
# â”€â”€ TRAIN DECISION TREE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# max_depth=4 keeps the tree readable; deeper trees tend to overfit
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)   # decision trees don't need scaling

print('Decision Tree trained.')
print()

# Feature importances
importance_df = pd.DataFrame({
    'Feature': SELECTED_FEATURES,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Decision Tree feature importances:')
print(importance_df.to_string(index=False))

In [ ]:
# Visualize the decision tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    dt_model,
    feature_names=SELECTED_FEATURES,
    class_names=['Late', 'On Time'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=False,
    proportion=True
)
ax.set_title('Decision Tree: The Rules the Model Learned\n'
             '(Blue = predicts On Time, Orange = predicts Late)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Follow the tree from top to bottom to see the if-then rules.')
print('At each node: go LEFT if the condition is TRUE, RIGHT if FALSE.')

---

## Part 3: Evaluating the Models  Why Accuracy Is Not Enough

Here is where the lesson becomes genuinely important.

The most natural way to evaluate a classification model is **accuracy** — what percentage of predictions were correct? But accuracy is a deceptive metric, especially when the classes are imbalanced.

In our dataset, 85.8% of submissions are on time. This means a model that **always predicts "on time"**  without looking at any features at all  would be 85.8% accurate. It would score better than many real models. And it would be completely useless.

To understand a model properly, we need to look at **what kinds of mistakes it makes**.

In [ ]:
# Generate predictions
lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]  # probability of on-time

dt_pred = dt_model.predict(X_test)
dt_prob = dt_model.predict_proba(X_test)[:, 1]

# Baseline: always predict on time
baseline_pred = np.ones(len(y_test), dtype=int)

print('=== Model Accuracy Comparison ===')
print(f'Always predict "on time" (baseline): {accuracy_score(y_test, baseline_pred)*100:.1f}%')
print(f'Logistic Regression:                 {accuracy_score(y_test, lr_pred)*100:.1f}%')
print(f'Decision Tree:                       {accuracy_score(y_test, dt_pred)*100:.1f}%')
print()
print('Does accuracy alone tell us which model is better?')

In [ ]:
# â”€â”€ CONFUSION MATRICES â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# The confusion matrix shows all four types of outcomes:
#
#  True Negative (TN):  Predicted LATE,    Actually LATE     âœ“ correct
#  False Positive (FP): Predicted ON TIME, Actually LATE     âœ— missed a late student
#  False Negative (FN): Predicted LATE,    Actually ON TIME  âœ— wrongly flagged on-time student
#  True Positive (TP):  Predicted ON TIME, Actually ON TIME  âœ“ correct

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = [
    ('Baseline\n(always "on time")', baseline_pred),
    ('Logistic Regression', lr_pred),
    ('Decision Tree', dt_pred),
]

for ax, (title, preds) in zip(axes, models):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Late (0)', 'On Time (1)']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{title}\nAccuracy: {acc*100:.1f}%', fontsize=10, fontweight='bold')

plt.suptitle('Confusion Matrices: Three Models Compared', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Rows = Actual class | Columns = Predicted class')

In [ ]:
# Detailed classification report for logistic regression
print('=== Logistic Regression: Full Classification Report ===')
print(classification_report(y_test, lr_pred, target_names=['Late (0)', 'On Time (1)']))

cm = confusion_matrix(y_test, lr_pred)
tn, fp, fn, tp = cm.ravel()

print('Breaking down the errors:')
print(f'  True Negatives  (predicted late, actually late):     {tn}')
print(f'  False Positives (predicted on-time, actually late):  {fp}  <-- MISSED late students')
print(f'  False Negatives (predicted late, actually on-time):  {fn}  <-- wrongly flagged students')
print(f'  True Positives  (predicted on-time, actually on-time): {tp}')
print()
print(f'The model missed {fp} out of {tn+fp} actually-late submissions ({fp/(tn+fp)*100:.0f}%).')
print(f'The model wrongly flagged {fn} students who actually submitted on time.')

### The Critical Finding

Look carefully at the confusion matrix. The model is 81% accurate  but it identifies only **1 out of 17 actually-late submissions** correctly. It misses 16 of them, predicting "on time" for students who were actually late.

This is the problem of **class imbalance**. Because 86% of submissions are on time, the model has learned that saying "on time" is almost always safe. It essentially defaults to the majority class and rarely flags anyone as at risk.

From a deployment perspective, this model would almost completely fail at its stated purpose  identifying students who are at risk of late submission. A teacher using this system would receive almost no useful alerts.

#### Two Types of Error  Two Different Harms

| Error Type | What Happened | Harm to Student |
|---|---|---|
| **False Positive** | Predicted "on time" but student was actually late | Student doesn't receive early support they needed |
| **False Negative** | Predicted "late" but student submitted on time | Student is wrongly flagged as at risk; may receive unwanted intervention or face unwarranted scrutiny |

Neither error is harmless. The question of which error is *worse* is a moral judgment, not a technical one.

> ðŸ“ **Reflection:** Which type of error concerns you more for this specific application  missing students who are actually late, or wrongly flagging students who are on time? What would the real-world consequences of each error be in a school setting?

*[Write your answer here]*

---

## Part 4: Better Evaluation  Precision, Recall, and ROC AUC

Since accuracy misleads us here, we need better metrics.

In [ ]:
# â”€â”€ DEFINING BETTER METRICS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#
# For the class we care most about (Late = 0):
#
# RECALL (Sensitivity):
#   Of all students who were actually late, what fraction did we catch?
#   recall = TP_late / (TP_late + FN_late)
#   We want this HIGH  missing late students is the primary failure.
#
# PRECISION:
#   Of all students we flagged as late, what fraction were actually late?
#   precision = TP_late / (TP_late + FP_late)
#   We want this HIGH  wrongly flagging on-time students wastes resources
#   and harms those students.
#
# ROC AUC:
#   Area under the ROC curve. Measures the model's ability to rank
#   predictions correctly. 1.0 = perfect, 0.5 = no better than random.

lr_auc = roc_auc_score(y_test, lr_prob)
dt_auc = roc_auc_score(y_test, dt_prob)

print('=== Better Metrics ===')
print()
print(f'Logistic Regression ROC AUC: {lr_auc:.3f}')
print(f'Decision Tree ROC AUC:       {dt_auc:.3f}')
print(f'(Random baseline AUC = 0.500)')
print()

# Recall for the "late" class specifically
cm_lr = confusion_matrix(y_test, lr_pred)
cm_dt = confusion_matrix(y_test, dt_pred)

def late_recall(cm):
    tn, fp, fn, tp = cm.ravel()
    # For the "late" class: tn = correct late, fp = late predicted on-time
    return tn / (tn + fp) if (tn + fp) > 0 else 0

print(f'Recall for "late" class (how many late submissions we caught):')
print(f'  Logistic Regression: {late_recall(cm_lr)*100:.1f}%')
print(f'  Decision Tree:       {late_recall(cm_dt)*100:.1f}%')
print(f'  Baseline (always on-time): 0.0%')

In [ ]:
#  ROC CURVE
# The ROC curve shows the trade-off between true positive rate and false
# positive rate at different prediction thresholds.
# A model that hugs the top-left corner is better.
# The diagonal line represents a random guesser.

fig, ax = plt.subplots(figsize=(7, 6))

for label, prob, auc, color in [
    ('Logistic Regression', lr_prob, lr_auc, '#5C8BC7'),
    ('Decision Tree', dt_prob, dt_auc, '#81C784'),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, label=f'{label} (AUC = {auc:.3f})', linewidth=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline (AUC = 0.500)')
ax.set_xlabel('False Positive Rate (wrongly flagged on-time students)')
ax.set_ylabel('True Positive Rate (late students correctly caught)')
ax.set_title('ROC Curve: How Well Can the Model Separate\nOn-Time from Late Submissions?',
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# CROSS-VALIDATION
# A single 80/20 split might be lucky or unlucky.
# 5-fold cross-validation splits the data 5 different ways and averages the results.
# This gives us a more reliable estimate of true performance.

print('=== 5-Fold Cross-Validation ===')
print('(Each number is from a different 80/20 split)')
print()

lr_cv = cross_val_score(
    LogisticRegression(random_state=42, max_iter=1000),
    scaler.fit_transform(X), y, cv=5, scoring='accuracy'
)
dt_cv = cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X, y, cv=5, scoring='accuracy'
)

print(f'Logistic Regression: {lr_cv.round(3)} â†’ mean {lr_cv.mean():.3f} Â± {lr_cv.std():.3f}')
print(f'Decision Tree:       {dt_cv.round(3)} â†’ mean {dt_cv.mean():.3f} Â± {dt_cv.std():.3f}')
print(f'Baseline:            0.858 (always predict on time)')

---

## Part 5: Understanding the Predictions — Who Gets It Wrong?

Aggregate metrics tell us how the model performs on average. But AI ethics requires us to look at individual cases — especially the ones the model gets wrong. Who are the students the model misses or mislabels?

In [ ]:
# Build a results dataframe combining predictions with the original rows.
# This keeps analysis columns available even when they were not model features.
RESULT_DISPLAY_COLS = [
    'student_id',
    'assignment_type',
    'difficulty',
    'days_until_due',
    'prior_completion_rate',
    'prior_avg_grade',
]

missing_display_cols = [col for col in RESULT_DISPLAY_COLS if col not in df.columns]
if missing_display_cols:
    raise KeyError(f'Missing expected dataset columns: {missing_display_cols}')

results = df.loc[X_test.index, RESULT_DISPLAY_COLS].copy()
results['actual']           = y_test.values
results['predicted']        = lr_pred
results['prob_on_time']     = lr_prob.round(3)
results['outcome'] = results.apply(
    lambda r: 'Correct' if r['actual'] == r['predicted'] else
              ('Missed late' if r['actual'] == 0 else 'Wrongly flagged'), axis=1
)

print('Summary of prediction outcomes:')
print(results['outcome'].value_counts())
print()

In [ ]:
# Students the model MISSED  actually late, predicted on time
# If this cell is run after an older results cell, add the display columns now.
if 'RESULT_DISPLAY_COLS' not in globals():
    RESULT_DISPLAY_COLS = [
        'student_id',
        'assignment_type',
        'difficulty',
        'days_until_due',
        'prior_completion_rate',
        'prior_avg_grade',
    ]

missing_display_cols = [col for col in RESULT_DISPLAY_COLS if col not in df.columns]
if missing_display_cols:
    raise KeyError(f'Missing expected dataset columns: {missing_display_cols}')

missing_result_cols = [col for col in RESULT_DISPLAY_COLS if col not in results.columns]
if missing_result_cols:
    results = results.join(df.loc[results.index, missing_result_cols])

missed = results[results['outcome'] == 'Missed late'].sort_values('prob_on_time', ascending=False)
print('Students the model MISSED (actually late, predicted on time):')
print('These students needed support, the AI gave them none.')
print()
print(missed[RESULT_DISPLAY_COLS + ['prob_on_time']].to_string(index=False))

In [ ]:
# Students the model WRONGLY FLAGGED  on time, predicted late
# If this cell is run after an older results cell, add the display columns now.
if 'RESULT_DISPLAY_COLS' not in globals():
    RESULT_DISPLAY_COLS = [
        'student_id',
        'assignment_type',
        'difficulty',
        'days_until_due',
        'prior_completion_rate',
        'prior_avg_grade',
    ]

missing_display_cols = [col for col in RESULT_DISPLAY_COLS if col not in df.columns]
if missing_display_cols:
    raise KeyError(f'Missing expected dataset columns: {missing_display_cols}')

missing_result_cols = [col for col in RESULT_DISPLAY_COLS if col not in results.columns]
if missing_result_cols:
    results = results.join(df.loc[results.index, missing_result_cols])

flagged = results[results['outcome'] == 'Wrongly flagged'].sort_values('prob_on_time')
print('Students WRONGLY FLAGGED (on time, predicted late):')
print('These students did fine, the AI suggested they were at risk.')
print()
print(flagged[RESULT_DISPLAY_COLS + ['prob_on_time']].to_string(index=False))

In [ ]:
# Distribution of prediction probabilities
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: distribution of predicted probabilities
for label, color, name in [(1, '#4CAF50', 'Actually on time'), (0, '#E57373', 'Actually late')]:
    subset = results[results['actual'] == label]['prob_on_time']
    axes[0].hist(subset, bins=15, alpha=0.65, color=color,
                 label=f'{name} (n={len(subset)})')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
axes[0].set_xlabel('Predicted probability of being on time')
axes[0].set_ylabel('Number of submissions')
axes[0].set_title('Predicted Probabilities by Actual Outcome', fontweight='bold')
axes[0].legend(fontsize=9)

# Right: outcome breakdown
outcome_counts = results['outcome'].value_counts()
colors_map = {'Correct': '#4CAF50', 'Missed late': '#E57373', 'Wrongly flagged': '#FFB74D'}
bar_colors = [colors_map[o] for o in outcome_counts.index]
axes[1].bar(outcome_counts.index, outcome_counts.values, color=bar_colors)
axes[1].set_title('Prediction Outcomes on Test Set', fontweight='bold')
axes[1].set_ylabel('Number of predictions')
for i, v in enumerate(outcome_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Why the Model Is Overconfident

Look at the left chart. Notice that the model rarely assigns low probabilities to anyone  most predictions cluster in the 0.7 - 1.0 range. The model has learned that almost everyone submits on time, so it almost always predicts high probabilities of on-time submission.

This **overconfidence** means the default decision threshold of 0.5 catches very few late students. You could lower the threshold  flag any student with a probability below 0.9 as "at risk"  but this would dramatically increase false positives (wrongly flagging students).

This is the fundamental **precision-recall trade-off**: you can catch more late students, but only by flagging more on-time students incorrectly. There is no free lunch.

> **Reflection:** If you were a teacher deploying this system, what threshold would you set, and why? What would it mean to flag 40% of students as "at risk" in order to catch 80% of the genuinely late ones?

*[Write your answer here]*

## Part 6: Adjusting the Threshold

The default decision threshold is 0.5: predict "late" if the model assigns less than 50% probability of on-time submission. But this threshold is arbitrary. Let's explore what happens when we change it.

In [ ]:
# Explore different decision thresholds
thresholds = [0.50, 0.70, 0.80, 0.85, 0.90, 0.95]

print('{:<12} {:<12} {:<14} {:<18} {}'.format('Threshold', 'Accuracy', 'Late recall', 'False flag rate', '% flagged as late'))
print('-' * 72)

for thresh in thresholds:
    preds = (lr_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_test, preds)
    late_rec = tn / (tn + fp) if (tn + fp) > 0 else 0
    false_flag_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
    flagged_pct = (preds == 0).mean()
    print(f'{thresh:<12.2f} {acc:<12.3f} {late_rec:<14.3f} {false_flag_rate:<18.3f} {flagged_pct:.1%}')

print()
print('Trade-off: Higher threshold catches more late students but flags more on-time students.')

In [ ]:
# Visualize the precision-recall trade-off
from sklearn.metrics import precision_recall_curve

# We want precision/recall for the LATE class (class 0)
# sklearn's precision_recall_curve works on the positive class,
# so we use 1 - prob_on_time as the score for "lateness"
lr_prob_late = 1 - lr_prob
precision, recall, thresholds_pr = precision_recall_curve(1 - y_test, lr_prob_late)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Precision-recall curve for late class
axes[0].plot(recall, precision, color='#5C8BC7', linewidth=2)
axes[0].set_xlabel('Recall (fraction of late students caught)')
axes[0].set_ylabel('Precision (fraction of flagged students truly late)')
axes[0].set_title('Precision-Recall Trade-Off\n(for the "Late" class)', fontweight='bold')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])
axes[0].axhline(17/100, color='gray', linestyle='--', linewidth=1,
                label=f'Baseline precision: {17/100:.2f}')
axes[0].legend(fontsize=9)

# Show catch rate vs false flag rate at different thresholds
catch_rates, false_flag_rates = [], []
for thresh in np.arange(0.4, 1.0, 0.01):
    preds = (lr_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    catch_rates.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    false_flag_rates.append(fn / (fn + tp) if (fn + tp) > 0 else 0)

axes[1].plot(false_flag_rates, catch_rates, color='#E57373', linewidth=2)
axes[1].set_xlabel('False Flag Rate (on-time students wrongly flagged)')
axes[1].set_ylabel('Late Catch Rate (late students correctly caught)')
axes[1].set_title('Catch Rate vs. False Flag Rate\nAs Threshold Changes', fontweight='bold')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

---

## Part 7: What the Model Results Mean Ethically

You have now built, trained, and evaluated a real AI model. Let's be honest about what it can and cannot do.

### What the Model Can Do
- Make predictions that are right about 81â€“83% of the time
- Identify that `prior_completion_rate` is strongly predictive
- With adjusted thresholds, flag more late-risk students  but at the cost of flagging many on-time students too

### What the Model Cannot Do
- Explain *why* any individual student submitted late
- Account for circumstances it has no data on: illness, family crises, learning differences
- Change its prediction for a student who has genuinely changed
- Feel compassion, exercise mercy, or recognize a student as a person rather than a pattern

### The Catholic Judgment

*Magnifica Humanitas* (no. 102) warns specifically about systems in high-stakes contexts that "do not know compassion, mercy, forgiveness, and above all, the hope that people are able to change." This is not a poetic statement  it is a precise description of what this model lacks and what any model lacks.

The Compendium of the Social Doctrine of the Church insists that the human person is always the **principle, subject, and purpose** of institutions  never their object. A homework prediction system that routes a student toward intervention, scrutiny, or reduced support based on their past record without any human judgment applied to their individual circumstances has violated this principle, regardless of its accuracy rate.

In [ ]:
# Final model summary for your project report
print('=' * 65)
print('MODEL EVALUATION REPORT  Module 3')
print('=' * 65)
print()
print(f'Model type:          Logistic Regression')
print(f'Features used:       {SELECTED_FEATURES}')
print(f'Training set size:   {len(X_train)}')
print(f'Test set size:       {len(X_test)}')
print()
print(f'Test accuracy:       {accuracy_score(y_test, lr_pred)*100:.1f}%')
print(f'ROC AUC:             {lr_auc:.3f}')
print(f'5-fold CV accuracy:  {lr_cv.mean():.3f} Â± {lr_cv.std():.3f}')
print()
tn, fp, fn, tp = confusion_matrix(y_test, lr_pred).ravel()
print(f'Late students correctly caught: {tn} / {tn+fp} ({tn/(tn+fp)*100:.0f}%)')
print(f'On-time students wrongly flagged: {fn} / {fn+tp} ({fn/(fn+tp)*100:.0f}%)')
print()
print('Most influential feature:        prior_completion_rate')
print('Feature most pushing toward late: days_until_due (negative coef)')
print()
print('ETHICAL FLAGS:')
print('  prior_completion_rate  feedback loop risk')
print('  prior_avg_grade        may encode socioeconomic advantage')
print('  Model catches only', f'{tn/(tn+fp)*100:.0f}%', 'of late submissions at 0.5 threshold')
print('=' * 65)

---

## Summary and What's Coming Next

### What You Accomplished in This Module

- Prepared data for modeling: encoding, scaling, and a principled train/test split
- Trained two interpretable models  logistic regression and decision tree  and understood what each learned
- Discovered that **81% accuracy is misleading** when the model misses 94% of the late students it should flag
- Learned to evaluate models using confusion matrices, recall, precision, and ROC AUC
- Explored the precision-recall trade-off: catching more late students always comes at the cost of flagging more on-time students
- Identified the specific students the model got wrong  and considered what that means for real people

### Coming in Module 4 (Classes 7-8)

Module 4 will focus on **bias auditing**: investigating whether the model performs equally well across different student groups and assignment types, or whether it systematically fails for some students more than others. This is where the ethical analysis deepens significantly  a model that is 81% accurate overall may be 95% accurate for some students and 60% accurate for others, and those differences are not random.

---

> **A thought to carry forward:** The model you built today flags approximately 19 students as "at risk of submitting late." Of those 19, only 1 is actually late. The other 18 submitted on time. If a teacher acted on every flag — perhaps following up with students, contacting parents, or adding notes to files — 18 out of 19 of those actions would be directed at students who did nothing wrong. What does that mean for how cautiously this system should be deployed — if at all?